# 6.1 感知机

机器学习的最终目的是找到一组良好的参数𝜃，使得𝜃表示的数学模型能够很好地从训
练集中学到映射关系𝑓𝜃: 𝒙 → 𝒚,   𝒙,𝒚 ∈ 𝔻train，从而利用训练好的𝑓𝜃
(𝒙), 𝒙 ∈ 𝔻𝑡𝑒𝑠𝑡去预测新
样本。神经网络属于机器学习的一个研究分支，它特指利用多个神经元去参数化映射函数
𝑓𝜃的模型。

感知机模型的结构如图 6.1 所示，它接受长度为𝑛的一维向量𝒙 = [𝑥1, 𝑥2, … , 𝑥𝑛]，每个  
输入节点通过权值为𝑤𝑖, 𝑖𝜖[1, 𝑛]的连接汇集为变量𝑧，即：  

𝑧 = 𝑤1 𝑥1 + 𝑤2 𝑥2 + ⋯ + 𝑤𝑛 𝑥𝑛 + b

# 6.2 全连接层

感知机模型的不可导特性严重约束了它的潜力，使得它只能解决极其简单的任务。实
际上，现代深度学习动辄数百万甚至上亿的参数规模，但它的核心结构与感知机并没有多
大差别。它在感知机的基础上，将不连续的阶跃激活函数换成了其它平滑连续可导的激活
函数，并通过堆叠多个网络层来增强网络的表达能力。
本节我们通过替换感知机的激活函数，同时并行堆叠多个神经元来实现多输入、多输
出的网络层结构。如图 6.4 所示，并行堆叠了 2 个神经元，即 2 个替换了激活函数的感知
机，构成 3 输入节点、2 个输出节点的网络层。

## 6.2.1 张量方式实现

在 TensorFlow 中，要实现全连接层，只需要定义好权值张量𝑾和偏置张量𝒃，并利用
TensorFlow 提供的批量矩阵相乘函数 tf.matmul()即可完成网络层的计算。  
例如，创建输入𝑿矩阵为𝑏 = 2个样本，每个样本的输入**特征长度**为𝑑in = 784，输出节点数为𝑑out = 256，故
定义权值矩阵𝑾的 shape 为[784,256]，并采用正态分布初始化𝑾；偏置向量𝒃的 shape 定义
为[256]，在计算完𝑿@𝑾后相加即可，最终全连接层的输出𝑶的 shape 为[2,256]，即 2 个样
本的特征，每个特征长度为 256，代码实现如下

In [3]:
import tensorflow as tf

# 创建 W,b 张量
x = tf.random.truncated_normal([2, 784])

w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros([256]))

o1 = tf.matmul(x, w1) + b1  # 线性变换
o1 = tf.nn.relu(o1)
o1.shape

TensorShape([2, 256])

## 6.2.2 层方式实现 layers.Dense(units, activation)  
全连接层本质上是矩阵的相乘和相加运算，实现并不复杂。但是作为最常用的网络层
之一，TensorFlow 中有更高层、使用更方便的层实现方式：layers.Dense(units, activation)。
通过 layer.Dense 类，只需要指定输出节点数 Units 和激活函数类型 activation 即可。需要注
意的是，输入节点数会根据第一次运算时的输入 shape 确定，同时根据输入、输出节点数
自动创建并初始化权值张量𝑾和偏置张量𝒃，因此在新建类 Dense 实例时，并不会立即创
建权值张量𝑾和偏置张量𝒃，而是需要调用 build 函数或者直接进行一次前向计算，才能完
成网络参数的创建。其中 activation 参数指定当前层的激活函数，可以为常见的激活函数或
自定义激活函数，也可以指定为 None，即无激活函数

In [6]:
x = tf.random.normal([4, 28 * 28])
layers = tf.keras.layers
# 创建全连接层，指定输出节点数和激活函数
fc = layers.Dense(512, activation=tf.nn.relu)
h1 = fc(x)  # 通过 fc 类实例完成一次全连接层的计算，返回输出张量
h1

<tf.Tensor: shape=(4, 512), dtype=float32, numpy=
array([[0.        , 0.9709538 , 3.3850536 , ..., 0.        , 0.32426673,
        0.        ],
       [0.6301449 , 0.        , 0.        , ..., 0.        , 1.1202126 ,
        1.3048234 ],
       [1.4610013 , 0.        , 0.        , ..., 1.047963  , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.8302963 , 0.67305994,
        0.        ]], dtype=float32)>

上述通过一行代码即可以创建一层全连接层 fc，并指定输出节点数为 512，输入的节点数
在fc(x)计算时自动获取，并创建内部权值张量𝑾和偏置张量𝒃。我们可以通过类内部的成
员名 kernel 和 bias 来获取权值张量𝑾和偏置张量𝒃对象

In [ ]:
fc.kernel  # 获取 Dense 类的权值矩阵

<tf.Variable 'dense/kernel:0' shape=(784, 512) dtype=float32, numpy=
array([[-0.02403864,  0.01718483,  0.03303132, ...,  0.01888648,
        -0.01189939,  0.01976474],
       [ 0.01844862, -0.03199852, -0.06070933, ...,  0.06471151,
         0.02184431, -0.01702722],
       [-0.0352652 ,  0.01177613,  0.03437458, ..., -0.01866053,
         0.01159595,  0.02554517],
       ...,
       [ 0.00735235, -0.05030629, -0.02205401, ..., -0.00135534,
         0.03997742,  0.06786095],
       [-0.01994214, -0.02163817, -0.00669475, ..., -0.03824362,
         0.00324025,  0.06358477],
       [-0.04635221, -0.00830283,  0.01597235, ..., -0.01033173,
        -0.03201895, -0.02569979]], dtype=float32)>

In [ ]:
fc.bias  # 获取Dense类的偏置向量

<tf.Variable 'dense/bias:0' shape=(512,) dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0.

可以看到，权值张量𝑾和偏置张量𝒃的 shape 和内容均符合我们的理解。  
在优化参数时，需要获得网络的所有**待优化的张量**参数列表，可以通过类的trainable_variables 来返回待优化参数列表，代码如下

In [10]:
fc.trainable_variables
# 返回待优化参数列表

[<tf.Variable 'dense/kernel:0' shape=(784, 512) dtype=float32, numpy=
 array([[-0.02403864,  0.01718483,  0.03303132, ...,  0.01888648,
         -0.01189939,  0.01976474],
        [ 0.01844862, -0.03199852, -0.06070933, ...,  0.06471151,
          0.02184431, -0.01702722],
        [-0.0352652 ,  0.01177613,  0.03437458, ..., -0.01866053,
          0.01159595,  0.02554517],
        ...,
        [ 0.00735235, -0.05030629, -0.02205401, ..., -0.00135534,
          0.03997742,  0.06786095],
        [-0.01994214, -0.02163817, -0.00669475, ..., -0.03824362,
          0.00324025,  0.06358477],
        [-0.04635221, -0.00830283,  0.01597235, ..., -0.01033173,
         -0.03201895, -0.02569979]], dtype=float32)>,
 <tf.Variable 'dense/bias:0' shape=(512,) dtype=float32, numpy=
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

实际上，网络层除了保存了待优化张量列表 trainable_variables，还有部分层包含了不
参与梯度优化的张量，如后续介绍的 Batch Normalization 层，可以通过
non_trainable_variables 成员返回所有不需要优化的参数列表。如果希望获得所有参数列
表，可以通过类的 variables 返回所有内部张量列表，

In [11]:
fc.variables

[<tf.Variable 'dense/kernel:0' shape=(784, 512) dtype=float32, numpy=
 array([[-0.02403864,  0.01718483,  0.03303132, ...,  0.01888648,
         -0.01189939,  0.01976474],
        [ 0.01844862, -0.03199852, -0.06070933, ...,  0.06471151,
          0.02184431, -0.01702722],
        [-0.0352652 ,  0.01177613,  0.03437458, ..., -0.01866053,
          0.01159595,  0.02554517],
        ...,
        [ 0.00735235, -0.05030629, -0.02205401, ..., -0.00135534,
          0.03997742,  0.06786095],
        [-0.01994214, -0.02163817, -0.00669475, ..., -0.03824362,
          0.00324025,  0.06358477],
        [-0.04635221, -0.00830283,  0.01597235, ..., -0.01033173,
         -0.03201895, -0.02569979]], dtype=float32)>,
 <tf.Variable 'dense/bias:0' shape=(512,) dtype=float32, numpy=
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

对于全连接层，内部张量都参与梯度优化，故 variables 返回的列表与 trainable_variables 相
同。  
利用网络层类对象进行前向计算时，只需要调用类的__call__方法即可，即写成 fc(x)
方式便可，它会自动调用类的__call__方法，在__call__方法中会自动调用 call 方法，这一
设定由 TensorFlow 框架自动完成，因此用户只需要将网络层的前向计算逻辑实现在 call 方
法中即可。对于全连接层类，在 call 方法中实现𝜎(𝑿@𝑾 + 𝒃)的运算逻辑，非常简单，最
后返回全连接层的输出张量即可。

# 6.3 神经网络  
通过层层堆叠图 6.4 中的全连接层，保证前一层的输出节点数与当前层的输入节点数
匹配，，即可堆叠出任意层数的网络。我们把这种由神经元相互连接而成的网络叫做神经网
络。如图 6.5 所示，通过堆叠 4 个全连接层，可以获得层数为 4 的神经网络，由于每层均为全连接层，称为全连接网络。其中第 1~3 个全连接层在网络中间，称之为隐藏层 1、2、
3，最后一个全连接层的输出作为网络的输出，称为输出层。隐藏层 1、2、3 的输出节点数
分别为[256,128,64]，输出层的输出节点数为 10。
在设计全连接网络时，网络的结构配置等超参数可以按着经验法则自由设置，只需要
遵循少量的约束即可。例如，隐藏层 1 的输入节点数需和数据的实际特征长度匹配，每层
的输入层节点数与上一层输出节点数匹配，输出层的激活函数和节点数需要根据任务的具
体设定进行设计。总的来说，神经网络模型的结构设计自由度较大，如图 6.5 层中每层的
输出节点数不一定要
[512,64,32,10]等都是可行的。
设计为[256
至于与哪一组超参数是最优的，这需要很多的
,128,64,10]，可以自由搭配，如[256,256,64,10
领域经验
]或
知识
和大量的实验尝试，或者可以通过 AutoML 技术搜索出较优设定

## 6.3.1 张量方式实现

对于多层神经网络，以图 6.5 网络结构为例，需要分别定义各层的权值矩阵𝑾和偏置
向量𝒃。有多少个全连接层，则需要相应地定义数量相当的𝑾和𝒃，并且每层的参数只能用
于对应的层，不能混淆使用。图 6.5 的网络模型实现如下：

In [15]:
# 隐藏层1张量
w1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
b1 = tf.Variable(tf.zeros([256]))
# 隐藏层2张量
w2 = tf.Variable(tf.random.truncated_normal([256, 128], stddev=0.1))
b2 = tf.Variable(tf.zeros([128]))
# 隐藏层3张量
w3 = tf.Variable(tf.random.truncated_normal([128, 64], stddev=0.1))
b3 = tf.Variable(tf.zeros([64]))
# 输出层张量
w4 = tf.Variable(tf.random.truncated_normal([64, 10], stddev=0.1))
b4 = tf.Variable(tf.zeros([10]))

在计算时，只需要按照网络层的顺序，将上一层的输出作为当前层的输入即可，重复
直至最后一层，并将输出层的输出作为网络的输出，代码如下：

In [26]:
with tf.GradientTape() as tape:  # 梯度记录器
    # x: [b, 28*28]
    # 隐藏层 1 前向计算，[b, 28*28] => [b, 256]
    h1 = x @ w1 + tf.broadcast_to(b1, [x.shape[0], 256])
    h1 = tf.nn.relu(h1)
    # 隐藏层 2 前向计算，[b, 256] => [b, 128]
    h2 = h1 @ w2 + tf.broadcast_to(b2, [h1.shape[0], 128])
    h2 = tf.nn.relu(h2)
    # 隐藏层 3 前向计算，[b, 128] => [b, 64]
    h3 = h2 @ w3 + tf.broadcast_to(b3, [h2.shape[0], 64])
    h3 = tf.nn.relu(h3)
    # 输出层前向计算，[b, 64] => [b, 10]
    h4 = h3 @ w4 + b4

最后一层是否需要添加激活函数通常视具体的任务而定，这里加不加都可以。
在使用 TensorFlow 自动求导功能计算梯度时，需要将前向计算过程放置在
tf.GradientTape()环境中，从而利用 GradientTape 对象的 gradient()方法自动求解参数的梯
度，并利用 optimizers 对象更新参数

## 6.3.2 层方式实现  
对于常规的网络层，通过层方式实现起来更加简洁高效。首先新建各个网络层类，并指定各层的激活函数类型

In [27]:
Sequential = tf.keras.Sequential

fc1 = layers.Dense(256, activation=tf.nn.relu)  # 隐藏层1
fc2 = layers.Dense(128, activation=tf.nn.relu)  # 隐藏层 2
fc3 = layers.Dense(64, activation=tf.nn.relu)  # 隐藏层 3
fc4 = layers.Dense(10, activation=None)  # 输出层

在前向计算时，依序通过各个网络层即可，代码如下：

In [32]:
x = tf.random.normal([4, 28 * 28])
h1 = fc1(x)  # 通过隐藏层1得到输出
h2 = fc2(h1)  # 通过隐藏层2得到输出
h3 = fc3(h2)  # ...
h4 = fc4(h3)  # 通过输出层得到网络输出
h4.shape

TensorShape([4, 10])

对于这种数据依次向前传播的网络，也可以通过 *Sequential* 容器封装成一个网络大类对象，  
调用大类的前向计算函数一次即可完成所有层的前向计算，使用起来更加方便，实现如下

In [34]:
Sequential = tf.keras.Sequential

# 通过 Sequential 容器封装为一个网络类
model = Sequential(
    [
        layers.Dense(256, activation=tf.nn.relu),
        layers.Dense(128, activation=tf.nn.relu),
        layers.Dense(64, activation=tf.nn.relu),
        layers.Dense(10, activation=None)
    ]
)
# 前向计算时只需要调用一次网络大类对象，即可完成所有层的按序计算
out = model(x)
out.shape

TensorShape([4, 10])

## 6.3.3 优化目标  


# 6.4 激活函数

## 6.4.1 Sigmoid  
它的一个优良特性就是能够把𝑥 ∈ 𝑅的输入“压缩”到𝑥 ∈ (0,1)区间，这个区间的数值在机
器学习常用来表示以下意义  
在 TensorFlow 中，可以通过 tf.nn.sigmoid 实现 Sigmoid 函数

In [35]:
x = tf.linspace(-6., 6., 10)
x

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-6.       , -4.6666665, -3.3333333, -2.       , -0.6666665,
        0.666667 ,  2.       ,  3.333334 ,  4.666667 ,  6.       ],
      dtype=float32)>

In [ ]:
tf.nn.sigmoid(x)  # 通过 Sigmoid 函数

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.00247262, 0.00931596, 0.0344452 , 0.11920292, 0.33924365,
       0.6607564 , 0.8807971 , 0.96555483, 0.99068403, 0.99752736],
      dtype=float32)>

## 6.4.2 ReLU  
ReLU 对小于 0 的值全部抑制为 0；对于正数则直接输
出，这种单边抑制特性来源于生物学.

在 TensorFlow 中，可以通过 tf.nn.relu 实现 ReLU 函数，代码如下

In [37]:
tf.nn.relu(x)  # 通过relu激活函数

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.      , 0.      , 0.      , 0.      , 0.      , 0.666667,
       2.      , 3.333334, 4.666667, 6.      ], dtype=float32)>

## LeakyReLU  
ReLU 函数在𝑥 < 0时导数值恒为 0，也可能会造成梯度弥散现象，为了克服这个问
题，LeakyReLU 函数被提出

In [39]:
tf.nn.leaky_relu(x, alpha=0.1)

<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([-0.6       , -0.46666667, -0.33333334, -0.2       , -0.06666666,
        0.666667  ,  2.        ,  3.333334  ,  4.666667  ,  6.        ],
      dtype=float32)>